### Comparing to Mattergen

- Convert the dataset used in the pre-train benefits study to a format that can be used by mattergen
- Preserves the splits etc

In [ ]:
import __init__
import datasets
from _utils import process_dataframe, filter_df_to_context

# load the dataset c-bone/mpdb-2prop_clean
dataset = datasets.load_dataset("c-bone/mpdb-2prop_clean")

# load as a dataframe
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

dfs = {
    "df_train": df_train,
    "df_val": df_val
}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

    print(f"length before filtering: {len(df)}")
    df = filter_df_to_context(df=df, context=1024, num_workers=64)
    print(f"after filtering: {len(df)}")

    df_clean = process_dataframe(df=df, num_workers=64, column_name="CIF")

    df_clean = df_clean.rename(columns={
        'CIF': 'cif',
        'Database': 'material_id',
        'Bandgap (eV)': 'dft_band_gap',
        'Energy Above Hull (eV)': 'energy_above_hull'
    })
    df_clean['material_id'] = [f"id_{i}" for i in range(len(df_clean))]
    df_clean = df_clean[["material_id", "cif", "dft_band_gap", "energy_above_hull"]]
    df_clean.to_csv(f'/home/cyprien/mattergen/datasets/mp_bandgap/{name}.csv', index=False)

### Training

- Mattergen repo was cloned, the new bandgap dataset was added to the repo
- Then made some custom non-default configs which can be found in `_config_files/training/conditional/pretraining_benefits/mattergen_configs`
- All the other configs stayed the same

> Note: There's a few differences compared to our pipeline to consider during comparison
> 1. The model trains on Reduced Niggli Structures whereas we train on Conventional CIFs
> 2. We train from scratch with CIF + Condition pairs whereas Mattergen has to do a base unconditional training, and then we need to do fine-tuning with adapter modules to allow for conditional generation
> 3. We take mattergen's mp-20 configuration and adapt it for this dataset because it is the most similar config from their repository available

> For point 3, this means:
> - Mattergen unconditional is trained for 900 epochs, our from scratch model is trained on around 23 epochs (with early stopping)
> - They use a batch size of 512, whereas ours is 32
> - They use ReduceLROnPlateau Schedule, whereas we use a CosineDecayWithMinLR
> - Training time is vastly different. 900 epochs is around 80h on 3x40GB GPUs
> - vs 23 epochs around 2h30m on 2x24GB GPUs

During generation, mattergen sets the distribution of N_ATOMS during generation to the one seen in training. So, we need to change this dsitribution in the code to the one in our dataset. Its calculated as below:

In [ ]:
from _utils._notebook_utils.b1b_mattergen_utils import compute_atom_counts, load_count_datasets, compute_atom_count_distribution

loaded = load_count_datasets([("MP Bandgap", "HF-databases/mpdb_processing/mpdb_2prop_count.parquet", 1024)])
loaded = compute_atom_counts(loaded)
atom_count_distributions = compute_atom_count_distribution(loaded[0][1])

print(atom_count_distributions)

### Results

In [ ]:
!python _utils/_metrics/VUN_metrics.py \
    --input_parquet '/home/cyprien/mattergen/gen_tests/test3_L_uncond/generated_crystals_cif.parquet' \
    --huggingface_dataset 'c-bone/mpdb-2prop_clean' \
    --load_processed_data 'HF-databases/mpdb-2prop_clean/mpdb_2prop_proc.parquet' \
    --output_parquet '/home/cyprien/mattergen/gen_tests/test3_L_uncond/generated_crystals_cif_metrics.parquet' \
    --num_workers 32 \
    --allow_stated_p1_mismatch

In [ ]:
!python _utils/_metrics/mace_ehull.py \
    --post_parquet /home/cyprien/mattergen/gen_tests/test3_L_uncond/generated_crystals_cif_metrics.parquet \
    --output_parquet /home/cyprien/mattergen/gen_tests/test3_L_uncond/generated_crystals_cif_metrics.parquet \
    --mp_data 'mp_computed_structure_entries.json.gz' \
    --num_workers 16